In [1]:
import numpy as np

def fspl_db(d_km, f_mhz):
    return 20 * np.log10(d_km) + 20 * np.log10(f_mhz) + 32.44

Pt_dbm = 43.0
Gt_db = 15.0
Gr_db = 8.0

distances = [0.1, 0.5, 1, 2, 5, 10]

for f in (900, 1800):
    print(f"\nFrequency: {f} MHz")
    print(f"{'Distance(km)':<15}{'FSPL(dB)':<12}{'Pr(dBm)':<12}{'Pr(W)':<14}")

    for d in distances:
        L = fspl_db(d, f)
        Pr_dbm = Pt_dbm + Gt_db + Gr_db - L
        Pr_w = 10 ** ((Pr_dbm - 30) / 10)

        print(f"{d:<15.1f}{L:<12.2f}{Pr_dbm:<12.2f}{Pr_w:<14.3e}")


Frequency: 900 MHz
Distance(km)   FSPL(dB)    Pr(dBm)     Pr(W)         
0.1            71.52       -5.52       2.802e-04     
0.5            85.50       -19.50      1.121e-05     
1.0            91.52       -25.52      2.802e-06     
2.0            97.55       -31.55      7.006e-07     
5.0            105.50      -39.50      1.121e-07     
10.0           111.52      -45.52      2.802e-08     

Frequency: 1800 MHz
Distance(km)   FSPL(dB)    Pr(dBm)     Pr(W)         
0.1            77.55       -11.55      7.006e-05     
0.5            91.52       -25.52      2.802e-06     
1.0            97.55       -31.55      7.006e-07     
2.0            103.57      -37.57      1.751e-07     
5.0            111.52      -45.52      2.802e-08     
10.0           117.55      -51.55      7.006e-09     


In [2]:
import math

S = 420                 # Total duplex channels available
M = 10                  # Number of cluster replications
n = 4                   # Path-loss exponent

valid = sorted({i*i + i*j + j*j
                for i in range(0, 5)
                for j in range(0, 5)
                if i*i + i*j + j*j > 0})

print(f"{'N':<8}{'Q (D/R)':<12}{'K':<8}{'Capacity':<12}{'S/I (dB)':<12}")

for N in valid:
    Q = math.sqrt(3 * N)        # Co-channel reuse ratio
    k = S // N                  # Channels per cell
    C = M * k * N               # System capacity

    SI = (Q ** n) / 6           # Signal-to-interference ratio
    SI_db = 10 * math.log10(SI)

    print(f"{N:<8}{Q:<12.2f}{k:<8}{C:<12}{SI_db:<12.2f}")

req = 18.0                    # AMPS requirement (dB)

for N in valid:
    if 10 * math.log10((math.sqrt(3*N) ** n) / 6) >= req:
        print(f"\nSmallest cluster size meeting S/I >= {req} dB: N = {N}")
        break

N       Q (D/R)     K       Capacity    S/I (dB)    
1       1.73        420     4200        1.76        
3       3.00        140     4200        11.30       
4       3.46        105     4200        13.80       
7       4.58        60      4200        18.66       
9       5.20        46      4140        20.85       
12      6.00        35      4200        23.34       
13      6.24        32      4160        24.04       
16      6.93        26      4160        25.84       
19      7.55        22      4180        27.34       
21      7.94        20      4200        28.21       
27      9.00        15      4050        30.39       
28      9.17        15      4200        30.70       
37      10.54       11      4070        33.12       
48      12.00       8       3840        35.39       

Smallest cluster size meeting S/I >= 18.0 dB: N = 7


In [3]:
import numpy as np

rng = np.random.default_rng(7)

D = 2000.0
P0, n, sigma = -38.0, 3.5, 4.0

x = np.arange(10, D, 5.0)

def rss(d, shadow):
    return P0 - 10 * n * np.log10(d) + shadow

r1 = rss(x, rng.normal(0, sigma, x.size))
r2 = rss(D - x, rng.normal(0, sigma, x.size))

def run(H):
    serving, handoffs, log = 1, 0, []

    for xi, a, b in zip(x, r1, r2):
        if serving == 1 and b > a + H:
            serving, handoffs = 2, handoffs + 1
            log.append((float(xi), '1->2'))

        elif serving == 2 and a > b + H:
            serving, handoffs = 1, handoffs + 1
            log.append((float(xi), '2->1'))

    return handoffs, log

print(f"{'H (dB)':>7}{'Handoffs':>10}{'Ping-pongs':>12}")

for H in (0, 2, 4, 6, 8, 10):
    h, log = run(H)
    print(f"{H:7}{h:10}{len(log):12}")

 H (dB)  Handoffs  Ping-pongs
      0        35          35
      2        27          27
      4        11          11
      6         7           7
      8         3           3
     10         1           1


In [4]:
import numpy as np

# Orthogonal Walsh codes
W = np.array([
    [1, 1, 1, 1],
    [1, -1, 1, -1],
    [1, 1, -1, -1],
    [1, -1, -1, 1]
])

# Data bits of three users
bits = {
    1: [1, 0, 1, 1],
    2: [0, 0, 1, 0],
    3: [1, 1, 0, 1]
}

L = W.shape[1]
nbits = 4

# Convert bits to BPSK
def bpsk(b):
    return 2 * np.array(b) - 1

# Spreading
tx = np.zeros(nbits * L)

for u, b in bits.items():
    spread = np.repeat(bpsk(b), L) * np.tile(W[u], nbits)

    print(f"User {u} data : {b}")
    print(f"User {u} spread : {spread.astype(int)}")

    tx += spread

print("\nComposite transmitted signal:\n", tx.astype(int))

# Despreading
print("\nDespreading at receiver:")

for u, b in bits.items():
    corr = (tx * np.tile(W[u], nbits)).reshape(nbits, L).sum(axis=1) / L

    rec = (corr > 0).astype(int)

    print(f"User {u} correlator: {corr} -> bits {rec.tolist()} "
          f"{'OK' if rec.tolist() == b else 'ERROR'}")

User 1 data : [1, 0, 1, 1]
User 1 spread : [ 1 -1  1 -1 -1  1 -1  1  1 -1  1 -1  1 -1  1 -1]
User 2 data : [0, 0, 1, 0]
User 2 spread : [-1 -1  1  1 -1 -1  1  1  1  1 -1 -1 -1 -1  1  1]
User 3 data : [1, 1, 0, 1]
User 3 spread : [ 1 -1 -1  1  1 -1 -1  1 -1  1  1 -1  1 -1 -1  1]

Composite transmitted signal:
 [ 1 -3  1  1 -1 -1 -1  3  1  1  1 -3  1 -3  1  1]

Despreading at receiver:
User 1 correlator: [ 1. -1.  1.  1.] -> bits [1, 0, 1, 1] OK
User 2 correlator: [-1. -1.  1. -1.] -> bits [0, 0, 1, 0] OK
User 3 correlator: [ 1.  1. -1.  1.] -> bits [1, 1, 0, 1] OK


In [5]:
import numpy as np

rng = np.random.default_rng(1)

U, TTI, NB = 6, 2000, 1

mean_snr_db = np.array([25, 20, 15, 10, 5, 0])
mean_snr = 10 ** (mean_snr_db / 10)

rate = np.log2(1 + mean_snr * rng.exponential(1.0, (TTI, U)))

def jain(x):
    return x.sum() ** 2 / (len(x) * (x ** 2).sum())

def simulate(policy):
    served = np.zeros(U)
    avg = np.ones(U)

    for t in range(TTI):
        if policy == "RR":
            u = t % U

        elif policy == "MaxCI":
            u = np.argmax(rate[t])

        else:
            u = np.argmax(rate[t] / avg)

        served[u] += rate[t, u]

        inst = np.zeros(U)
        inst[u] = rate[t, u]

        avg = 0.99 * avg + 0.01 * inst

    return served / TTI

print(f"{'Scheduler':<10}", end=" ")

for i in range(U):
    print(f"U{i+1:>4}", end=" ")

print("| Total | Jain")

for p in ("RR", "MaxCI", "PF"):
    th = simulate(p)

    print(f"{p:<10}", end=" ")

    for v in th:
        print(f"{v:6.3f}", end=" ")

    print(f"| {th.sum():6.3f} | {jain(th):.3f}")

Scheduler  U   1 U   2 U   3 U   4 U   5 U   6 | Total | Jain
RR          1.235  0.996  0.703  0.479  0.297  0.152 |  3.861 | 0.741
MaxCI       6.061  1.630  0.199  0.015  0.002  0.000 |  7.906 | 0.264
PF          1.689  1.375  1.038  0.741  0.476  0.257 |  5.576 | 0.778
